In [1]:
import pytz
from datetime import datetime, date, timedelta
import pandas as pd
import altair as alt
import altair_latimes as lat

In [2]:
alt.themes.register('latimes', lat.theme)
alt.themes.enable('latimes')

ThemeRegistry.enable('latimes')

In [3]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)

### Import

In [4]:
df = pd.read_csv(
    "../data/raw/bor/colorado-river-reservoir-bor-timeseries.csv", 
    parse_dates=["date"]
)

In [5]:
hist_df = pd.read_csv(
    "../data/metadata/reservoirs-historical-averages.csv"
)

In [6]:
details_df = pd.read_csv(
    "../data/metadata/reservoirs-metadata-details.csv"
)

### Clean

In [7]:
df.columns = df.columns.str.lower()

In [8]:
df.columns = df.columns.str.replace(" ","_")

In [9]:
df = df.rename(columns={
    "station_id": "reservoir_id",
    "date_time": "date",
    "storage": "storage_af"
})

### Filter to last 365 days from today

In [10]:
today = datetime.today().date()
today

datetime.date(2026, 10, 27)

In [11]:
last_year = (today - pd.DateOffset(days=365)).date()
last_year

datetime.date(2025, 10, 27)

Trim to last 365 days and remove invalid values

In [12]:
trim_df = df[
    (df.date >= pd.to_datetime(last_year))
].copy()

### Merge historical average

Create month column for merging

In [13]:
trim_df["month"] = pd.DatetimeIndex(trim_df.date).month

In [14]:
month_names = {
    "January": 1,
    "February": 2,
    "March": 3,
    "April": 4,
    "May": 5,
    "June": 6,
    "July": 7,
    "August": 8,
    "September": 9,
    "October": 10,
    "November": 11,
    "December": 12,
}

In [15]:
hist_df["month"] = hist_df["month"].map(month_names)

In [16]:
merge_historical = pd.merge(
    trim_df[["reservoir_id","date","month","storage_af"]],
    hist_df[["reservoir_id", "month", "average_storage_value", "average_storage_unit"]],
    how="left",
    on=["reservoir_id","month"]
)

In [17]:
merge_details = pd.merge(
    merge_historical,
    details_df[["id", "lake", "capacity", "lat", "lon"]],
    how="left",
    left_on=["reservoir_id"],
    right_on=["id"]
).drop(["id", "average_storage_unit"], axis=1)

### Calculate percentages

In [18]:
merge_details["current_level_pct_of_total"] = merge_details["storage_af"] / merge_details["capacity"]

In [19]:
merge_details["average_level_pct_of_total"] = merge_details["average_storage_value"] / merge_details["capacity"]

In [20]:
merge_details["current_level_pct_of_avg"] = merge_details["storage_af"] / merge_details["average_storage_value"]

In [21]:
merge_details

,reservoir_id,date,month,storage_af,average_storage_value,lake,capacity,lat,lon,current_level_pct_of_total,average_level_pct_of_total,current_level_pct_of_avg
0,MHV,2025-10-27,10,1614027.340,1483843,Lake Mohave,1810000.000,35.197000°,-114.567000°,0.892,0.820,1.088
1,MHV,2025-10-28,10,1618652.260,1483843,Lake Mohave,1810000.000,35.197000°,-114.567000°,0.894,0.820,1.091
2,MHV,2025-10-29,10,1619496.080,1483843,Lake Mohave,1810000.000,35.197000°,-114.567000°,0.895,0.820,1.091
3,MHV,2025-10-30,10,1617345.850,1483843,Lake Mohave,1810000.000,35.197000°,-114.567000°,0.894,0.820,1.090
4,MHV,2025-10-31,10,1613701.180,1483843,Lake Mohave,1810000.000,35.197000°,-114.567000°,0.892,0.820,1.088
...,...,...,...,...,...,...,...,...,...,...,...,...
1435,HVS,2026-10-17,10,539585.670,569207,Lake Havasu,648000.000,34.317000°,-114.156000°,0.833,0.878,0.948
1436,HVS,2026-10-18,10,536076.440,569207,Lake Havasu,648000.000,34.317000°,-114.156000°,0.827,0.878,0.942
1437,HVS,2026-10-19,10,534909.650,569207,Lake Havasu,648000.000,34.317000°,-114.156000°,0.825,0.878,0.940
1438,HVS,2026-10-20,10,533821.520,569207,Lake Havasu,648000.000,34.317000°,-114.156000°,0.824,0.878,0.938


### Drop missing values

In [22]:
drop_na_df = merge_details.dropna(subset=["storage_af"])

### Chart

In [23]:
melt = pd.melt(
    drop_na_df,
    id_vars=["date","lake"],
    value_vars=["current_level_pct_of_total", "average_level_pct_of_total"]
)

In [24]:
# alt.data_transformers.disable_max_rows()

# alt.Chart(melt).mark_line().encode(
#     x='date:T',
#     y='value',
#     color='variable',
#     column='lake'
# ).properties(
#     width=180,
#     height=180
# )

### Export

Trim and rename for export

In [25]:
export_df = drop_na_df[
    ['reservoir_id',
     'lake', 
     'date', 
     'storage_af', 
     'average_storage_value', 
     'capacity', 
     'current_level_pct_of_total',
     'average_level_pct_of_total', 
     'current_level_pct_of_avg', 
     'lat', 
     'lon' ]
].rename(columns={
    'lake': 'reservoir_name',
    'average_storage_value': 'historical_average',
    'capacity': 'total_capacity'
}).sort_values("total_capacity", ascending=False)

In [26]:
export_df.to_csv("../data/processed/bor/colorado-river-reservoirs-timeseries.csv", index=False)

In [27]:
latest_df = export_df[export_df.date == export_df.date.max()]

In [28]:
latest_df.to_csv(f"../data/processed/bor/colorado-latest.csv", index=False)